# RAG hỏi đáp thông tin thuốc – Google Colab

Hệ thống có **2 nguồn kiến thức**:
1. `thuoc_48.csv`: thông tin 48 thuốc (hoạt chất, chỉ định, liều dùng, chống chỉ định, tác dụng phụ, lưu ý). Mỗi thuốc được cắt thành nhiều đoạn theo mục.
2. `tong_hop_48_thuoc_cau_hoi_test.csv`: các cặp câu hỏi – câu trả lời mẫu (cột `cau_hoi`, `cau_tra_loi`).

Bạn gõ câu hỏi, hệ thống tìm các đoạn liên quan nhất rồi để mô hình sinh câu trả lời tiếng Việt kèm nguồn.

**Chuẩn bị:** upload 2 file CSV lên `MyDrive/NCKH/`, bật GPU (Runtime → Change runtime type → T4 GPU), rồi chạy lần lượt từng cell từ trên xuống.
Cell 1–7 chạy 1 lần. Cell 8 là khung hỏi đáp. Cell 9–10 là đo độ chính xác (tuỳ chọn, để lấy số liệu báo cáo).

## Cell 1 – Cài thư viện

In [1]:
!pip install -q sentence-transformers faiss-cpu transformers accelerate pandas

## Cell 2 – Kết nối Google Drive

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Cell 3 – Cấu hình (sửa đường dẫn nếu file của bạn để chỗ khác)

In [3]:
import os, re, torch
import numpy as np
import pandas as pd

FILE_THUOC = "/content/drive/MyDrive/NCKH/thuoc_48.csv"
FILE_QA    = "/content/drive/MyDrive/NCKH/bo_tra_loi_cau_hoi.csv"

EMBED_MODEL = "BAAI/bge-m3"                # embedding hỗ trợ tiếng Việt
LLM_MODEL   = "Qwen/Qwen2.5-3B-Instruct"   # chạy được trên T4 miễn phí
# Muốn câu trả lời tốt hơn: "Qwen/Qwen2.5-7B-Instruct" (cần thêm load_in_4bit=True ở Cell 6)

TOP_K_DOAN = 6     # số đoạn thông tin thuốc đưa cho mô hình
TOP_K_QA   = 3     # số cặp hỏi-đáp mẫu đưa cho mô hình
DUNG_QA    = True  # True: dùng thêm cặp hỏi-đáp mẫu khi trả lời (dùng thật). Phần đo độ chính xác tự tắt để không "gian lận".

for p in (FILE_THUOC, FILE_QA):
    print("[OK]" if os.path.exists(p) else "[CẢNH BÁO] Không thấy file:", p)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "KHÔNG CÓ - hãy bật GPU")

[OK] /content/drive/MyDrive/NCKH/thuoc_48.csv
[OK] /content/drive/MyDrive/NCKH/bo_tra_loi_cau_hoi.csv
GPU: Tesla T4


## Cell 4 – Đọc 2 file CSV, cắt đoạn, tạo embedding và FAISS index
- Mỗi thuốc được cắt thành các đoạn: *Thông tin chung* (hoạt chất, dạng bào chế), *Chỉ định*, *Liều dùng*, *Chống chỉ định*, *Tác dụng phụ*, *Lưu ý*. Mỗi đoạn đều ghi rõ tên thuốc ở đầu, để tìm kiếm không bị nhầm thuốc.
- Các cặp hỏi-đáp mẫu được đánh chỉ mục theo **câu hỏi**.

In [4]:
import pandas as pd
import torch
import faiss
from sentence_transformers import SentenceTransformer

def doc_csv(path):
    # Thử lần lượt các encoding phổ biến cho file CSV tiếng Việt
    for enc in ("utf-8-sig", "cp1258", "cp1252", "latin-1"):
        try:
            d = pd.read_csv(path, encoding=enc).fillna("")
            print(f"Đọc OK: {path} | encoding = {enc}")
            return d
        except UnicodeDecodeError:
            continue
    raise ValueError(f"Không đọc được file: {path}")

df  = doc_csv(FILE_THUOC)
dqa = doc_csv(FILE_QA)
print(f"File thuốc: {len(df)} dòng | cột: {list(df.columns)}")
print(f"File hỏi-đáp: {len(dqa)} dòng | cột: {list(dqa.columns)}")

def sach(v):
    v = str(v).strip()
    return "" if (not v or v.startswith("<")) else v

# ---- Cắt mỗi thuốc thành các đoạn theo mục ----
MUC = [  # (nhãn, [các cột gộp vào đoạn này])
    ("Thông tin chung",  ["hoat_chat", "dang_bao_che"]),
    ("Chỉ định",         ["chi_dinh"]),
    ("Liều dùng",        ["lieu_dung"]),
    ("Chống chỉ định",   ["chong_chi_dinh"]),
    ("Tác dụng phụ",     ["tac_dung_phu"]),
    ("Lưu ý",            ["luu_y"]),
]
NHAN_COT = {"hoat_chat":"Hoạt chất", "dang_bao_che":"Dạng bào chế"}

doan_van, doan_ten, doan_muc, doan_nguon = [], [], [], []
for _, r in df.iterrows():
    ten = sach(r.get("ten_thuoc", ""))
    if not ten:
        continue
    nguon = sach(r.get("nguon", ""))
    for nhan, cols in MUC:
        mau = []
        for c in cols:
            v = sach(r.get(c, ""))
            if v:
                mau.append(f"{NHAN_COT.get(c, nhan)}: {v}" if len(cols) > 1 else v)
        if mau:
            doan_van.append(f"Thuốc: {ten}\n{nhan}: " + "; ".join(mau) if len(cols) == 1 else f"Thuốc: {ten}\n" + "\n".join(mau))
            doan_ten.append(ten); doan_muc.append(nhan); doan_nguon.append(nguon)

ten_goc = list(dict.fromkeys(doan_ten))   # danh sách tên thuốc, không trùng
print(f"Số thuốc dùng được: {len(ten_goc)} | Số đoạn: {len(doan_van)}")

# ---- Cặp hỏi-đáp mẫu ----
qa_hoi  = [str(x).strip() for x in dqa["cau_hoi"]]
qa_dap  = [str(x).strip() for x in dqa["cau_tra_loi"]]
qa_ten  = [str(x).strip() for x in dqa["ten_thuoc"]]
qa_van  = [f"Câu hỏi: {h}\nTrả lời: {d}" for h, d in zip(qa_hoi, qa_dap)]

# ---- Embedding + FAISS ----
embedder = SentenceTransformer(EMBED_MODEL, device="cuda" if torch.cuda.is_available() else "cpu")

def ma_hoa(ds):
    return embedder.encode(ds, batch_size=16, show_progress_bar=True, normalize_embeddings=True).astype("float32")

v_doan = ma_hoa(doan_van)
idx_doan = faiss.IndexFlatIP(v_doan.shape[1]); idx_doan.add(v_doan)

v_qa = ma_hoa(qa_hoi)          # đánh chỉ mục theo CÂU HỎI để khớp với câu hỏi người dùng
idx_qa = faiss.IndexFlatIP(v_qa.shape[1]); idx_qa.add(v_qa)

print("Xong. FAISS đoạn thuốc:", idx_doan.ntotal, "| FAISS hỏi-đáp:", idx_qa.ntotal)

Đọc OK: /content/drive/MyDrive/NCKH/thuoc_48.csv | encoding = utf-8-sig
Đọc OK: /content/drive/MyDrive/NCKH/bo_tra_loi_cau_hoi.csv | encoding = utf-8-sig
File thuốc: 48 dòng | cột: ['ten_thuoc', 'hoat_chat', 'dang_bao_che', 'chi_dinh', 'lieu_dung', 'chong_chi_dinh', 'tac_dung_phu', 'luu_y', 'nguon']
File hỏi-đáp: 1824 dòng | cột: ['stt', 'ten_thuoc', 'cau_hoi', 'cau_tra_loi', 'nguon']
Số thuốc dùng được: 48 | Số đoạn: 288


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

... (11 outputs left out)

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

Batches:   0%|          | 0/18 [00:00<?, ?it/s]

Batches:   0%|          | 0/114 [00:00<?, ?it/s]

Xong. FAISS đoạn thuốc: 288 | FAISS hỏi-đáp: 1824


## Cell 5 – Kiểm tra tên thuốc 2 file có khớp nhau không

In [5]:
a = set(qa_ten); b = set(ten_goc)
print("Có trong file hỏi-đáp nhưng KHÔNG có trong file thuốc:", a - b or "không có (tốt)")
print("Có trong file thuốc nhưng KHÔNG có trong file hỏi-đáp:", b - a or "không có (tốt)")

Có trong file hỏi-đáp nhưng KHÔNG có trong file thuốc: không có (tốt)
Có trong file thuốc nhưng KHÔNG có trong file hỏi-đáp: không có (tốt)


## Cell 6 – Tải mô hình trả lời (LLM) – lần đầu mất vài phút

In [6]:
from transformers import AutoTokenizer, AutoModelForCausalLM

tok = AutoTokenizer.from_pretrained(LLM_MODEL)
llm = AutoModelForCausalLM.from_pretrained(LLM_MODEL, torch_dtype=torch.float16, device_map="auto")
print("Đã tải LLM:", LLM_MODEL)

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


... (2 outputs left out)

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Đã tải LLM: Qwen/Qwen2.5-3B-Instruct


## Cell 7 – Hàm tìm kiếm và trả lời
- Nếu câu hỏi **có nhắc tên thuốc** (ví dụ "Becacold-E"), hệ thống chỉ tìm trong thông tin của thuốc đó để không bị nhầm sang thuốc khác.
- Nếu không nhắc tên thuốc, hệ thống tìm trên toàn bộ 48 thuốc.

In [9]:
# ===== Các hàm tìm kiếm =====
def ten_co_ban(t):
    # "Coxileb 200mg" -> "coxileb"; "Becacold-E" -> "becacold-e"
    return re.sub(r"\s*\d.*$", "", t).strip().lower()

BASE = {t: ten_co_ban(t) for t in ten_goc}

def tim_ten_thuoc(cau_hoi):
    q = cau_hoi.lower()
    return [t for t, b in BASE.items() if b and b in q]

def truy_xuat(cau_hoi, k=TOP_K_DOAN, k_qa=TOP_K_QA, dung_qa=True, loc_ten=True):
    qv = embedder.encode([cau_hoi], normalize_embeddings=True).astype("float32")
    thuoc_nhac = tim_ten_thuoc(cau_hoi) if loc_ten else []

    # Đoạn thông tin thuốc
    D, I = idx_doan.search(qv, idx_doan.ntotal)
    doan = []
    for i, s in zip(I[0], D[0]):
        if thuoc_nhac and doan_ten[i] not in thuoc_nhac:
            continue
        doan.append((int(i), float(s)))
        if len(doan) == k:
            break

    # Cặp hỏi-đáp mẫu
    qa = []
    if dung_qa:
        D, I = idx_qa.search(qv, idx_qa.ntotal)
        for i, s in zip(I[0], D[0]):
            if thuoc_nhac and qa_ten[i] not in thuoc_nhac:
                continue
            qa.append((int(i), float(s)))
            if len(qa) == k_qa:
                break
    return doan, qa, thuoc_nhac

# ===== Cấu hình trả lời =====
NGUONG_CO_TEN    = 0.78   # câu hỏi đã nhắc tên thuốc
NGUONG_KHONG_TEN = 0.88   # câu hỏi không nhắc tên thuốc
BO_SUNG   = True          # True: thêm thông tin bổ sung sau câu trả lời của bạn
K_BO_SUNG = 4             # số đoạn thông tin thuốc đưa cho mô hình để bổ sung

def sinh(messages, max_tokens=450):
    text = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tok(text, return_tensors="pt").to(llm.device)
    with torch.no_grad():
        out = llm.generate(**inputs, max_new_tokens=max_tokens, do_sample=False, repetition_penalty=1.05)
    return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

def doan_cua_thuoc(cau_hoi, ten, k=K_BO_SUNG):
    # Lấy các đoạn liên quan nhất, nhưng CHỈ của đúng thuốc đã khớp
    qv = embedder.encode([cau_hoi], normalize_embeddings=True).astype("float32")
    D, I = idx_doan.search(qv, idx_doan.ntotal)
    kq = []
    for i, s in zip(I[0], D[0]):
        if doan_ten[i] == ten:
            kq.append((int(i), float(s)))
            if len(kq) == k:
                break
    return kq

def hoi(cau_hoi, dung_qa=None, in_ra=True, max_tokens=450):
    if dung_qa is None:
        dung_qa = DUNG_QA
    doan, qa, thuoc_nhac = truy_xuat(cau_hoi, k_qa=3, dung_qa=dung_qa)

    # ===== ƯU TIÊN 1: có câu hỏi tương ứng trong file -> câu trả lời của bạn + bổ sung =====
    nguong = NGUONG_CO_TEN if thuoc_nhac else NGUONG_KHONG_TEN
    if qa and qa[0][1] >= nguong:
        j, s = qa[0]
        goc = qa_dap[j]
        them = ""
        doan_bs = doan_cua_thuoc(cau_hoi, qa_ten[j]) if BO_SUNG else []

        if doan_bs:
            tai_lieu = "\n\n".join(f"[Đoạn {n+1}]\n{doan_van[i]}" for n, (i, _) in enumerate(doan_bs))
            messages = [
                {"role": "system", "content":
                 "Bạn là trợ lý cung cấp thông tin thuốc. Bạn được cho một CÂU TRẢ LỜI GỐC đã hoàn chỉnh. "
                 "Nhiệm vụ: viết PHẦN BỔ SUNG nối tiếp ngay sau câu trả lời gốc, dài 2 đến 3 câu. "
                 "Quy tắc: (1) chỉ viết thông tin MỚI, có trong TÀI LIỆU và liên quan trực tiếp đến câu hỏi; "
                 "(2) không lặp lại những gì câu trả lời gốc đã nói; (3) không bịa số liệu, liều dùng; "
                 "(4) giọng văn tự nhiên, nhẹ nhàng, giống câu trả lời gốc, viết liền mạch thành đoạn văn, "
                 "không gạch đầu dòng, không chào hỏi, không mở đầu lại; "
                 "(5) nếu TÀI LIỆU không có gì mới đáng bổ sung, chỉ trả lời đúng: [KHONG_CO]."},
                {"role": "user", "content":
                 f"CÂU HỎI: {cau_hoi}\n\nCÂU TRẢ LỜI GỐC:\n{goc}\n\nTÀI LIỆU:\n{tai_lieu}\n\nPHẦN BỔ SUNG:"},
            ]
            them = sinh(messages, max_tokens=220)
            if "[KHONG_CO]" in them or len(them) < 20:
                them = ""

        tra_loi = goc + (" " + them if them else "")
        if in_ra:
            print(f"\nTRẢ LỜI:\n{tra_loi}")
            print(f"\n(Khớp câu mẫu: \"{qa_hoi[j]}\" | độ giống {s:.3f} | "
                  f"{'có bổ sung' if them else 'không bổ sung'})")
        return tra_loi, doan_bs or doan, qa

    # ===== ƯU TIÊN 2: câu hỏi ngoài bộ khuôn -> mô hình viết theo giọng văn các câu mẫu =====
    ngu_canh = "\n\n".join(f"[Đoạn {n+1}]\n{doan_van[i]}" for n, (i, _) in enumerate(doan))
    messages = [{"role": "system", "content":
        "Bạn là trợ lý cung cấp thông tin thuốc. "
        "Nội dung chuyên môn CHỈ lấy từ phần TÀI LIỆU, không được bịa. "
        "Nếu tài liệu không có, nói rõ 'Tài liệu không có thông tin này'. "
        "Về CÁCH VIẾT: bắt chước sát giọng văn, cách mở đầu câu (ví dụ 'Thuốc X có...'), "
        "độ dài của các câu trả lời mẫu trong hội thoại trước. Viết thành đoạn văn liền mạch, tiếng Việt."}]
    for i, _ in reversed(qa):
        messages.append({"role": "user", "content": qa_hoi[i]})
        messages.append({"role": "assistant", "content": qa_dap[i]})
    messages.append({"role": "user", "content": f"TÀI LIỆU:\n{ngu_canh}\n\nCÂU HỎI: {cau_hoi}"})
    tra_loi = sinh(messages, max_tokens)

    if in_ra:
        print("\nTRẢ LỜI:\n" + tra_loi)
        print("\nNGUỒN:")
        for i, s in doan:
            print(f"  - {doan_ten[i]} | mục: {doan_muc[i]} | độ giống: {s:.3f} | {doan_nguon[i]}")
    return tra_loi, doan, qa

## Cell 8 – HỎI ĐÁP: gõ câu hỏi, nhận câu trả lời
Chạy cell này, gõ câu hỏi rồi nhấn Enter. Hệ thống trả lời và hỏi tiếp. Gõ `thoat` (hoặc Enter trống) để dừng.
Ví dụ: `Liều dùng của Becacold-E là bao nhiêu?`

In [10]:
while True:
    cau = input("\nNhập câu hỏi (gõ 'thoat' để dừng): ").strip()
    if not cau or cau.lower() == "thoat":
        print("Đã dừng.")
        break
    hoi(cau)


Nhập câu hỏi (gõ 'thoat' để dừng): zinc có gì bên trong

TRẢ LỜI:
Thuốc Zinc có chứa hoạt chất chính là Kẽm gluconat 70mg (tương đương 10mg Kẽm nguyên tố) và được bào chế dưới dạng viên nén. Sản phẩm này thường được sử dụng trong việc phòng ngừa và điều trị thiếu kẽm, hỗ trợ rối loạn tiêu hóa, tiêu chảy, và tăng cường đề kháng. Nó cũng có thể được chỉ định cho những người cần bổ sung kẽm do thiếu hụt hoặc nhu cầu đặc biệt.

NGUỒN:
  - Zinc | mục: Lưu ý | độ giống: 0.525 | Tờ hướng dẫn sử dụng Thuốc Zinc 70Mg DHG
  - Zinc | mục: Tác dụng phụ | độ giống: 0.521 | Tờ hướng dẫn sử dụng Thuốc Zinc 70Mg DHG
  - Zinc | mục: Thông tin chung | độ giống: 0.517 | Tờ hướng dẫn sử dụng Thuốc Zinc 70Mg DHG
  - Zinc | mục: Chỉ định | độ giống: 0.514 | Tờ hướng dẫn sử dụng Thuốc Zinc 70Mg DHG
  - Zinc | mục: Chống chỉ định | độ giống: 0.502 | Tờ hướng dẫn sử dụng Thuốc Zinc 70Mg DHG
  - Zinc | mục: Liều dùng | độ giống: 0.479 | Tờ hướng dẫn sử dụng Thuốc Zinc 70Mg DHG

Nhập câu hỏi (gõ 'thoat' để dừng

## Cell 9 (tuỳ chọn) – Đo độ chính xác TRUY XUẤT
Với mỗi câu hỏi trong file hỏi-đáp, kiểm tra thuốc đúng có nằm trong các đoạn tìm được không.
Phần này **tắt lọc theo tên thuốc và tắt cặp hỏi-đáp mẫu** để đo thuần khả năng tìm kiếm của embedding (không bị gian lận).

In [11]:
top1 = topk = 0
sai = []
for h, ten in zip(qa_hoi, qa_ten):
    doan, _, _ = truy_xuat(h, dung_qa=False, loc_ten=False)
    kq = [doan_ten[i] for i, _ in doan]
    top1 += (len(kq) > 0 and kq[0] == ten)
    topk += (ten in kq)
    if ten not in kq:
        sai.append((ten, h, list(dict.fromkeys(kq))))

n = len(qa_hoi)
print(f"Số câu hỏi: {n}")
print(f"Top-1 accuracy: {top1/n:.2%}")
print(f"Top-{TOP_K_DOAN} accuracy: {topk/n:.2%}")
print(f"Số câu truy xuất sai: {len(sai)}")
for ten, h, kq in sai[:10]:
    print(f"  - Cần: {ten} | Hỏi: {h} | Nhận được: {kq}")

Số câu hỏi: 1824
Top-1 accuracy: 99.40%
Top-6 accuracy: 100.00%
Số câu truy xuất sai: 0


## Cell 10 (tuỳ chọn) – Đo chất lượng CÂU TRẢ LỜI so với `cau_tra_loi`
Lấy ngẫu nhiên `SO_CAU` câu, để hệ thống trả lời (**không dùng cặp hỏi-đáp mẫu** để khỏi chép lại đáp án), rồi so với đáp án gốc bằng độ giống ngữ nghĩa (0 đến 1). Kết quả chi tiết lưu ra Drive để bạn đọc và chấm tay.
Mỗi câu mất khoảng vài chục giây; muốn chạy thử nhanh đặt `SO_CAU = 10`.

In [12]:
SO_CAU = 50
rng = np.random.RandomState(42)
chon = rng.choice(len(qa_hoi), size=min(SO_CAU, len(qa_hoi)), replace=False)

rows = []
for n_, i in enumerate(chon, 1):
    tra_loi, doan, _ = hoi(qa_hoi[i], dung_qa=False, in_ra=False)
    v = embedder.encode([tra_loi, qa_dap[i]], normalize_embeddings=True)
    rows.append({
        "ten_thuoc": qa_ten[i], "cau_hoi": qa_hoi[i],
        "dap_an_goc": qa_dap[i], "he_thong_tra_loi": tra_loi,
        "do_giong": round(float(np.dot(v[0], v[1])), 4),
    })
    if n_ % 10 == 0:
        print(f"  đã xong {n_}/{len(chon)} câu")

kq_df = pd.DataFrame(rows)
print(f"\nĐộ giống trung bình với đáp án gốc: {kq_df['do_giong'].mean():.3f}")
print(f"Tỉ lệ câu có độ giống >= 0.8: {(kq_df['do_giong'] >= 0.8).mean():.2%}")

DUONG_LUU = "/content/drive/MyDrive/NCKH/ket_qua_danh_gia.csv"
kq_df.to_csv(DUONG_LUU, index=False, encoding="utf-8-sig")
print("Đã lưu kết quả chi tiết:", DUONG_LUU)

  đã xong 10/50 câu
  đã xong 20/50 câu
  đã xong 30/50 câu
  đã xong 40/50 câu
  đã xong 50/50 câu

Độ giống trung bình với đáp án gốc: 0.888
Tỉ lệ câu có độ giống >= 0.8: 94.00%
Đã lưu kết quả chi tiết: /content/drive/MyDrive/NCKH/ket_qua_danh_gia.csv
